# Mouse pancreas: endocrine-lineage GRN analysis

Analyze cell-type-specific regulatory programs and their progression along pancreatic endocrinogenesis.

This notebook starts from the cell-specific GRNs produced by `main.py`. It does not retrain scDGRN. Large per-cell networks remain under `results/` and are intentionally excluded from Git.

**Workflow**

1. Validate the dataset and inferred-network outputs.
2. Represent every cell by its top regulatory edges.
3. quantify GRN similarity and recover cell groups.
4. summarize cell-type-specific transcription-factor activity.
5. perform dataset-specific developmental analysis.

Run the notebook from top to bottom. Set `SCDGRN_RESULTS_ROOT` or `SCDGRN_RESULT_NAME` before launching Jupyter when results are stored outside the default `results/pancreas/` directory.


## 1. Configure paths and analysis parameters


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the scDGRN repository.")


REPO_ROOT = find_repository_root()
sys.path.insert(0, str(REPO_ROOT))

DATASET_NAME = "pancreas"
RESULT_NAME = os.environ.get("SCDGRN_RESULT_NAME", "pancreas")
DATASETS_ROOT = Path(os.environ.get("SCDGRN_DATASETS_ROOT", REPO_ROOT / "datasets")).resolve()
RESULTS_ROOT = Path(os.environ.get("SCDGRN_RESULTS_ROOT", REPO_ROOT / "results")).resolve()
DATASET_DIR = DATASETS_ROOT / DATASET_NAME
RESULT_DIR = RESULTS_ROOT / RESULT_NAME
OUTPUT_DIR = RESULT_DIR / "tutorial_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "celltype"
RANDOM_SEED = 3407
TOP_N_EDGES = 50

print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATASET_DIR}")
print(f"Results:    {RESULT_DIR}")
print(f"Outputs:    {OUTPUT_DIR}")


## 2. Validate required inputs


In [ ]:
from tutorial.tutorial_utils import load_cell_labels, validate_analysis_inputs

paths = validate_analysis_inputs(DATASET_DIR, RESULT_DIR)
cell_labels = load_cell_labels(paths["labels"], LABEL_COLUMN)
tf_names = pd.read_csv(paths["tf_list"], header=None).iloc[:, 0].astype(str).to_numpy()

print(f"Cells: {len(cell_labels):,}")
print(f"Cell types: {cell_labels.nunique()}")
print(f"TFs: {len(tf_names):,}")
print(cell_labels.value_counts().to_string())


## 3. Build the cell-by-cell GRN similarity matrix

Each cell is represented by its top-weighted TF-target edges. Pairwise similarity is the Jaccard index between these edge sets. The matrix is cached because this is the most expensive common step.


In [ ]:
from tutorial.tutorial_utils import build_jaccard_similarity

similarity_path = OUTPUT_DIR / f"{DATASET_NAME}_top{TOP_N_EDGES}_jaccard.npy"
if similarity_path.exists():
    similarity = np.load(similarity_path)
    print(f"Loaded cached similarity matrix: {similarity_path}")
else:
    similarity = build_jaccard_similarity(
        RESULT_DIR / "single_network_tf",
        n_cells=len(cell_labels),
        top_n=TOP_N_EDGES,
        exclude_unit_weight=True,
    )
    np.save(similarity_path, similarity)
    print(f"Saved similarity matrix: {similarity_path}")

print("Similarity matrix shape:", similarity.shape)


## 4. Cluster cells from their inferred GRNs

The adjusted Rand index compares unsupervised GRN-derived clusters with the supplied cell annotations. It is a descriptive agreement measure and does not make the annotations ground truth for regulatory state.


In [ ]:
from sklearn.metrics import adjusted_rand_score
from tutorial.tutorial_utils import leiden_from_similarity

clusters = leiden_from_similarity(
    similarity,
    k=24,
    resolution=0.35,
    seed=RANDOM_SEED,
)
ari = adjusted_rand_score(cell_labels.to_numpy(), clusters)

cluster_table = pd.DataFrame({
    "cell_index": np.arange(len(cell_labels)),
    "cell_type": cell_labels.to_numpy(),
    "leiden_cluster": clusters,
})
cluster_table.to_csv(OUTPUT_DIR / f"{DATASET_NAME}_grn_clusters.csv", index=False)
print(f"Adjusted Rand index: {ari:.3f}")


In [ ]:
import umap

embedding = np.loadtxt(RESULT_DIR / "cell_embedding" / "cell_embedding.csv", delimiter="\t")
coords = umap.UMAP(n_neighbors=20, min_dist=0.25, random_state=RANDOM_SEED).fit_transform(embedding)

plot_df = pd.DataFrame({
    "UMAP1": coords[:, 0],
    "UMAP2": coords[:, 1],
    "cell_type": cell_labels.to_numpy(),
    "cluster": clusters.astype(str),
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
sns.scatterplot(data=plot_df, x="UMAP1", y="UMAP2", hue="cell_type", s=12, linewidth=0, ax=axes[0])
sns.scatterplot(data=plot_df, x="UMAP1", y="UMAP2", hue="cluster", palette="tab20", s=12, linewidth=0, ax=axes[1])
axes[0].set_title("Annotated cell types")
axes[1].set_title(f"GRN-derived Leiden clusters (ARI={ari:.3f})")
for ax in axes:
    ax.set(xticks=[], yticks=[])
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)

fig.savefig(OUTPUT_DIR / f"{DATASET_NAME}_grn_embedding.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / f"{DATASET_NAME}_grn_embedding.svg", bbox_inches="tight")
plt.show()


## 5. Compare GRN similarity between annotated cell types


In [ ]:
from tutorial.tutorial_utils import mean_similarity_by_group

cell_type_similarity = mean_similarity_by_group(similarity, cell_labels.to_numpy())

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cell_type_similarity, cmap="mako", square=True, ax=ax)
ax.set_title(f"Mean top-{TOP_N_EDGES} GRN similarity between cell types")
fig.savefig(OUTPUT_DIR / f"{DATASET_NAME}_cell_type_similarity.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / f"{DATASET_NAME}_cell_type_similarity.svg", bbox_inches="tight")
plt.show()

cell_type_similarity


## 6. Summarize transcription-factor regulatory activity

For each cell, TF activity is defined here as the sum of outgoing edge weights in its inferred GRN. The heatmap shows row-wise standardized cell-type means.


In [ ]:
from tutorial.tutorial_utils import compute_tf_activity

activity_path = OUTPUT_DIR / f"{DATASET_NAME}_tf_activity.npy"
if activity_path.exists():
    tf_activity = np.load(activity_path)
    print(f"Loaded cached TF activity: {activity_path}")
else:
    tf_activity = compute_tf_activity(
        RESULT_DIR / "single_network",
        n_cells=len(cell_labels),
        n_tfs=len(tf_names),
    )
    np.save(activity_path, tf_activity)
    print(f"Saved TF activity: {activity_path}")

activity_df = pd.DataFrame(tf_activity, columns=tf_names)
activity_df["cell_type"] = cell_labels.to_numpy()
mean_activity = activity_df.groupby("cell_type", observed=True).mean()

top_by_type = {
    cell_type: mean_activity.loc[cell_type].nlargest(5).index.tolist()
    for cell_type in mean_activity.index
}
display(pd.DataFrame.from_dict(top_by_type, orient="index", columns=[f"Top {i}" for i in range(1, 6)]))

selected_tfs = sorted({tf for values in top_by_type.values() for tf in values})
heatmap_data = mean_activity[selected_tfs].T
heatmap_data = heatmap_data.sub(heatmap_data.mean(axis=1), axis=0)
heatmap_data = heatmap_data.div(heatmap_data.std(axis=1).replace(0, 1), axis=0)

fig, ax = plt.subplots(figsize=(max(7, 0.7 * len(mean_activity)), max(5, 0.24 * len(selected_tfs))))
sns.heatmap(heatmap_data, cmap="vlag", center=0, ax=ax)
ax.set_xlabel("Cell type")
ax.set_ylabel("Transcription factor")
ax.set_title("Cell-type-specific TF regulatory activity")
fig.savefig(OUTPUT_DIR / f"{DATASET_NAME}_tf_activity_heatmap.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / f"{DATASET_NAME}_tf_activity_heatmap.svg", bbox_inches="tight")
plt.show()


## 7. Order endocrine cell states along pseudotime


In [ ]:
stage_order = ["Ngn3 low EP", "Ngn3 high EP", "Pre-endocrine", "Beta"]
pseudotime = pd.read_csv(DATASET_DIR / "pseudotime.csv")["pseudotime"].to_numpy(dtype=float)
trajectory = pd.DataFrame({
    "cell_index": np.arange(len(cell_labels)),
    "cell_type": cell_labels.to_numpy(),
    "pseudotime": pseudotime,
})
trajectory = trajectory.loc[trajectory["cell_type"].isin(stage_order)].copy()
trajectory["cell_type"] = pd.Categorical(trajectory["cell_type"], categories=stage_order, ordered=True)
trajectory = trajectory.sort_values(["pseudotime", "cell_index"], kind="mergesort")

display(trajectory.groupby("cell_type", observed=True)["pseudotime"].agg(["count", "min", "median", "max"]))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
sns.violinplot(data=trajectory, x="cell_type", y="pseudotime", order=stage_order, inner="quartile", cut=0, ax=ax)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
fig.savefig(OUTPUT_DIR / "pancreas_endocrine_stage_pseudotime.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "pancreas_endocrine_stage_pseudotime.svg", bbox_inches="tight")
plt.show()


Continue with [`pdx1_pseudotime.ipynb`](pdx1_pseudotime.ipynb) to reproduce the top-50 Pdx1-target regulatory-activity curves along the endocrine trajectory.


## Interpretation boundary

These analyses summarize associations inferred by scDGRN. High edge weights, GRN similarity and pseudotime trends do not by themselves establish direct biochemical regulation or causal cell-state transitions.
